harmonic series of clusters: 15.828 
## Important formula:
- K: The area of the largest cluster  
- N: The number of the clusters  
$$
\frac{K}{\ln K +\gamma(\approx 0.577)} N = 10^8  
$$

Check: the answer is {{Ans}}

In [ ]:
msg = "Hello world"
print(msg)
print(msg)
# clear_output()
from ipywidgets import interact

@interact(x=(0, 10))
def square(x=5):
    print(x * x)


In [ ]:
import glob
import re
import numpy as np
import matplotlib as plt
import plotly.graph_objects as go
import math
from ipywidgets import interact
from IPython.display import clear_output


In [ ]:
# Initialize lists to store data
Size = 3000
current_z_list = []
avg_cluster_list = []
avg_cluster_size_list = []
avg_cluster_portion_list = []

# Find the file that starts with "output_cluster_Size" in the specified directory
file_list = glob.glob("./output_cluster_Size=3000*")

# Read and parse data from each file
for file_path in file_list:
    with open(file_path, 'r') as file:
        content = file.read()
        
        # Split the content into blocks based on newline character
        blocks = content.split('\n\n')  # Split based on double newline for blocks
        
        for block in blocks:
            current_z_match = re.search(r'Current Z = (.+)', block)
            avg_cluster_match = re.search(r'Average number of clusters: (.+)', block)
            avg_cluster_size_match = re.search(r'Average number of clusters by size: (.+)', block)
            avg_cluster_portion_match = re.search(r'Average portion of clusters by size: (.+)', block)
            
            if current_z_match and avg_cluster_match and avg_cluster_size_match and avg_cluster_portion_match:
                current_z = float(current_z_match.group(1))
                avg_cluster = float(avg_cluster_match.group(1))
                avg_cluster_size = np.array([float(x) for x in avg_cluster_size_match.group(1).split()])
                avg_cluster_portion = np.array([float(x) for x in avg_cluster_portion_match.group(1).split()])
                for i in range(len(avg_cluster_size)):
                    avg_cluster_portion[i] = avg_cluster_size[i]*(i+1)
                # Append data to lists
                current_z_list.append(current_z)
                avg_cluster_list.append(avg_cluster)
                avg_cluster_size_list.append(avg_cluster_size)
                avg_cluster_portion_list.append(avg_cluster_portion)
            else:
                print(f"Error parsing data from block in file: {file_path}")
# test of input                
# print("Z = ", current_z_list)
# print("number = ", avg_cluster_list)
# print(avg_cluster_size_list)
# print(avg_cluster_portion_list)


In [ ]:
# Define a function to update the plot based on selected Z
import time 
from IPython.display import clear_output


@interact(Z=(min(current_z_list), max(current_z_list), 0.01), continuous_update=False)

def update_plot(Z=min(current_z_list)):

#    if not plt.waitforbuttonpress():
#        return
    # Find the index corresponding to the selected Z
    idx = np.abs(np.array(current_z_list) - Z).argmin()
    
    # Create the Plotly figure
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.arange(1,len(avg_cluster_portion_list[idx])), y=avg_cluster_size_list[idx]/(Size*Size), mode='lines', name='Average size of Clusters (Blue)'))
    fig.add_trace(go.Scatter(x=np.arange(1, len(avg_cluster_portion_list[idx])), y=avg_cluster_portion_list[idx]/(Size*Size), mode='lines', name='Average Portion of Clusters (Red)', line=dict(color='red')))
    # Calculate and display the sum of average portion of clusters
    sum_avg_portion = np.sum(avg_cluster_portion_list[idx]/(Size*Size))
    fig.update_yaxes(type='log', range=[-10, 1]) 
    fig.update_xaxes(range=[pow(10,0), pow(10,4)])
    #fig.update_xaxes(type='log', range = [0,5])
    fig.add_annotation(text=f'Sum of Average Portion: {sum_avg_portion:.5f}', x=2, y=1, showarrow=False)
    
    # Update layout
    fig.update_layout(title=f'Z = {current_z_list[idx]}', xaxis_title='Number', yaxis_title='Average Portion')
    
    # Clear the previous plot
    # time.sleep(1)
    clear_output()
    
    # Show the plot
    fig.show()



In [ ]:
from IPython.display import clear_output
from ipywidgets import interact
import plotly.graph_objects as go
import ipywidgets as widgets

# Define a function to update the plot based on selected Z
@interact(Z=(min(current_z_list), max(current_z_list), 0.01), continuous_update=False)


def update_plot(Z=min(current_z_list)):
    # Create a button widget to trigger the update
    button = widgets.Button(description="Update Plot")
    
    # Define a function to update the plot when the button is clicked
    def on_button_clicked(b):
        # Find the index corresponding to the selected Z
        idx = np.abs(np.array(current_z_list) - Z).argmin()
        
        # Create the Plotly figure
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=np.arange(len(avg_cluster_portion_list[idx])), y=avg_cluster_portion_list[idx], mode='lines', name='Average Portion of Clusters'))
        
        # Calculate and display the sum of average portion of clusters
        sum_avg_portion = np.sum(avg_cluster_portion_list[idx])
        fig.add_annotation(text=f'Sum of Average Portion: {sum_avg_portion:.5f}', x=0, y=1, showarrow=False)
        
        # Update layout
        fig.update_layout(title=f'Z = {current_z_list[idx]}', xaxis_title='Number', yaxis_title='Average Portion')
        
        # Clear the previous plot
        # clear_output()
        
        # Show the plot
        fig.show()

    # Set the button's on_click event to the update function
    button.on_click(on_button_clicked)
    # Display the button
    display(button)



In [ ]:
from IPython.display import clear_output, display
from ipywidgets import interact
import plotly.graph_objects as go
import ipywidgets as widgets


# Define a function to update the plot based on selected Z
start_change = False
@interact(Z=(min(current_z_list), max(current_z_list), 0.01), continuous_update=False)

def update_plot(Z=min(current_z_list)):
    # Create a button widget to trigger the update
    button = widgets.Button(description="Update Plot")
    
    # Define a list to store the output widgets
    output_list = []
    # Define a function to update the plot when the button is clicked
    def on_button_clicked(_):   
        output = widgets.Output()
        output_list.append(output)
        global start_change
        # print(output_list)
        # print(start_change)
        with output:
            # Find the index corresponding to the selected Z
            idx = np.abs(np.array(current_z_list) - Z).argmin()
            
            # Create the Plotly figure
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=np.arange(len(avg_cluster_portion_list[idx])), y=avg_cluster_portion_list[idx], mode='lines', name='Average Portion of Clusters'))
            
            # Calculate and display the sum of average portion of clusters
            sum_avg_portion = np.sum(avg_cluster_portion_list[idx])
            fig.add_annotation(text=f'Sum of Average Portion: {sum_avg_portion:.5f}', x=0, y=1, showarrow=False)
            
            # Update layout
            fig.update_layout(title=f'Z = {current_z_list[idx]}', xaxis_title='Number', yaxis_title='Average Portion')
            # Show the plot
            display(fig)
            # output_list.pop(1)
    
    # Set the button's on_click event to the update function
    button.on_click(on_button_clicked) 
    # Display the button
    display(button)
